In [3]:
import sys
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu126
True


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision 
from torchvision.datasets import CIFAR10

In [5]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root = "./data", train = True, download = True, transform = transform)
testset = CIFAR10(root = "./data", train = False, download = True, transform = transform)

100%|████████████████████████████████████████████████████████████████████████████████| 170M/170M [08:42<00:00, 326kB/s]


In [6]:
trainloader = DataLoader(trainset, batch_size=64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

In [21]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [22]:
model =  CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [24]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f" Epoch = {epoch + 1}/ {epochs}, Training Loss = {epoch_training_loss/len(trainloader)}")

 Epoch = 1/ 10, Training Loss = 0.24516793625319705
 Epoch = 2/ 10, Training Loss = 0.1771735055467395
 Epoch = 3/ 10, Training Loss = 0.14259844880713068
 Epoch = 4/ 10, Training Loss = 0.12126003056669327
 Epoch = 5/ 10, Training Loss = 0.10520367223955214
 Epoch = 6/ 10, Training Loss = 0.09859502977689209
 Epoch = 7/ 10, Training Loss = 0.0916767031826136
 Epoch = 8/ 10, Training Loss = 0.0789167792813214
 Epoch = 9/ 10, Training Loss = 0.07835922081647512
 Epoch = 10/ 10, Training Loss = 0.07142917376340312


In [26]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy : {(correct_labels/total_labels) * 100}")

Accuracy : 74.3
